# Event Attention-Curve Clustering with Sliding SAX

**Revision note.** This notebook was revised to match the current project pipeline:

```text
Google Trends chunk files
        ↓
overlap-rolling stitched daily series
        ↓
quality diagnostics
        ↓
light denoising
        ↓
detrending
        ↓
robust per-series normalization
        ↓
sliding SAX parameter tuning
        ├── window_size
        ├── step_size
        ├── PAA word length w / n_segments
        └── alphabet_size a
        ↓
SAX feature matrix
        ↓
KMeans / Ward clustering
        ↓
cluster interpretation tables and plots
```

The notebook **does not re-query Google Trends**. It treats the existing overlap-rolling stitched files under
`DATA_DIR/<query>/stitched/` as the canonical daily input.

## External references used for this revision

1. `candicedjorno/replication-restoring-google-trends` motivates the preprocessing order: cluster or group related terms, smooth/denoise noisy Google Trends curves, then detrend before modeling. In this notebook, that logic is adapted to an unsupervised SAX clustering setting rather than forecasting.
   - Repository: https://github.com/candicedjorno/replication-restoring-google-trends
   - Relevant ideas/files: `scripts/01_clustering/hierarchical_clustering.ipynb`, `scripts/02_denoising/gt_denoising.R`, `scripts/03_detrending/detrending.py`

2. `trendecon/trendecon` motivates careful construction of consistent long daily Google Trends series. Since the current project has already downloaded chunk-level daily data and stitched it with overlap scaling, this notebook does **not** implement trendEcon's multi-frequency reconstruction. It instead adds diagnostics and post-stitch preprocessing.
   - Repository: https://github.com/trendecon/trendecon
   - Documentation: https://trendecon.github.io/trendecon/reference/ts_gtrends_mwd.html
   - Relevant idea: robust long daily Google Trends series from daily/weekly/monthly harmonization; here used as conceptual guidance, not as executable dependency.

## Main revision choices

- Removed the consensus-matrix / graph-clustering tuning from the core workflow.
- Kept only the three SAX tuning families requested:
  - `window_size`
  - `step_size`
  - `PAA word length w`, implemented as `n_segments`
  - `alphabet_size a`
- Separated **representation tuning** from **cluster-count selection**.
- Added robust preprocessing functions: small-gap interpolation, light denoising, detrending, and robust z-scoring.
- Added output notes so every selected parameter setting is saved for reporting.


## Efficiency revision added

This version also improves computation efficiency in the SAX tuning stage by:

- caching clean NumPy arrays once instead of repeatedly converting each pandas Series;
- caching sliding windows by `(window_size, step)` so the same windows are reused across PAA and alphabet settings;
- computing PAA segment means with cumulative sums instead of Python loops over every window;
- vectorizing SAX discretization with `np.searchsorted`;
- using sampled silhouette scores during grid search to avoid expensive pairwise-distance calculations on large term sets;
- using a smaller `n_init` for tuning and reserving heavier `KMeans` only for the final selected model.


In [2]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# CONFIG -- edit for your local machine
# -----------------------------------------------------------------------------

DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_06")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Current folder structure:
# DATA_DIR/
#     query_metadata.csv
#     weather/
#         chunks/
#         diagnostics/
#         stitched/
#             gt_stitched_weather_2022-01-01_2026-05-31.csv
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_stitched_*.csv"

# If your files use a different naming convention, these fallbacks are checked too.
STITCHED_FALLBACK_GLOBS = [
    "gt_stitched_fixed_*.csv",
    "gt_fixed_*.csv",
    "*.csv",
]

# Preprocessing controls.
MAX_INTERPOLATE_GAP = 7       # fill missing gaps up to one week
DENOISE_WINDOW = 15           # odd integer; light smoothing only
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91           # 13-week rolling median trend

# SAX tuning grid. PAA word length w is named n_segments in the code.
WINDOW_GRID = [60, 90, 180, 365]
STEP_FRACTIONS = [0.25]       # step = round(window_size * fraction)
PAA_WORD_LENGTH_GRID = [8, 12, 16, 24, 32, 52]
ALPHABET_GRID = [3, 4, 5, 6]

# Clustering grid. This is evaluated after a SAX representation is built.
K_GRID = range(2, 8)
RANDOM_STATE = 42

# Optional spike features summarize the full preprocessed curve in addition to symbolic shape.
INCLUDE_SPIKE_FEATURES = True

# Efficiency controls for tuning.
# Silhouette is O(n_terms^2); sampling makes grid search much faster on large panels.
TUNING_KMEANS_N_INIT = 10
FINAL_KMEANS_N_INIT = 50
SILHOUETTE_SAMPLE_SIZE = 500
USE_WINDOW_CACHE = True


## 1. Load stitched daily series

The file of interest is the stitched daily series produced from the `chunks/` folder. The raw chunk files remain important for diagnostics, but the clustering input is the stitched output.

This is the practical compromise between the two external references:

- trendEcon emphasizes that long Google Trends series need careful scaling across time windows.
- The current project already performed overlap-rolling stitching, so we avoid re-querying or re-normalizing every chunk.
- The Djorno replication motivates the next stage: statistical preprocessing before modeling/clustering.


In [3]:
def load_all_series(
    data_dir: Path,
    stitched_subdir: str = STITCHED_SUBDIR,
    filename_glob: str = STITCHED_GLOB,
) -> dict[str, pd.Series]:
    """
    Load one stitched daily Google Trends CSV per query folder.

    Expected format:
        DATA_DIR/<query>/stitched/gt_stitched_<query>_<start>_<end>.csv

    The first non-Time column is treated as the query's value column.
    """
    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []

    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir():
            continue

        stitched_dir = folder / stitched_subdir
        candidates = sorted(stitched_dir.glob(filename_glob))

        if not candidates:
            for pattern in STITCHED_FALLBACK_GLOBS:
                candidates = sorted(stitched_dir.glob(pattern))
                if candidates:
                    break

        if not candidates:
            failed.append((folder.name, f"missing stitched csv in {stitched_dir}"))
            continue

        fpath = candidates[-1]

        try:
            df = pd.read_csv(fpath, parse_dates=["Time"])
            value_cols = [c for c in df.columns if c != "Time"]
            if not value_cols:
                raise ValueError("no value column found")

            value_col = value_cols[0]
            ts = (
                df[["Time", value_col]]
                .dropna(subset=["Time"])
                .drop_duplicates("Time")
                .sort_values("Time")
                .set_index("Time")[value_col]
                .astype(float)
            )
            ts.name = folder.name
            series_dict[folder.name] = ts

        except Exception as e:
            failed.append((folder.name, str(e)))

    print(f"Loaded {len(series_dict)} stitched series.")
    if failed:
        print(f"Failed to load {len(failed)} folders:")
        for name, err in failed[:25]:
            print(f"  - {name}: {err}")
        if len(failed) > 25:
            print(f"  ... {len(failed) - 25} more")

    return series_dict


def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all query series on a common daily date index."""
    if not series_dict:
        raise ValueError("series_dict is empty")

    panel = pd.DataFrame(series_dict).sort_index()
    full_index = pd.date_range(panel.index.min(), panel.index.max(), freq="D")
    return panel.reindex(full_index)


def basic_quality_report(panel: pd.DataFrame) -> pd.DataFrame:
    """Basic post-stitch diagnostics per query."""
    rows = []
    for col in panel.columns:
        x = panel[col]
        rows.append({
            "query": col,
            "n_days_total": len(x),
            "n_nonmissing": int(x.notna().sum()),
            "missing_share": float(x.isna().mean()),
            "zero_share": float((x == 0).mean(skipna=True)),
            "min": float(x.min(skipna=True)),
            "max": float(x.max(skipna=True)),
            "mean": float(x.mean(skipna=True)),
            "std": float(x.std(skipna=True)),
            "range": float(x.max(skipna=True) - x.min(skipna=True)),
        })
    return pd.DataFrame(rows).sort_values("query")


## 2. Post-stitch preprocessing

The preprocessing layer adapts the Djorno et al. logic to this project:

1. fill short missing gaps;
2. apply light denoising, so SAX does not overreact to one-day noise;
3. remove slow-moving trend, so clustering focuses on attention shape;
4. robustly normalize each query once over the full daily series.

This normalization is **not per chunk** and **not per SAX window**. Window-level z-normalization still occurs inside SAX so each local word describes shape rather than absolute level.


In [4]:
def interpolate_small_gaps(s: pd.Series, max_gap: int = MAX_INTERPOLATE_GAP) -> pd.Series:
    """Interpolate short missing gaps without filling long absent regions."""
    return s.astype(float).interpolate(
        method="time",
        limit=max_gap,
        limit_direction="both",
    )


def denoise_series(
    s: pd.Series,
    window: int = DENOISE_WINDOW,
    polyorder: int = DENOISE_POLYORDER,
) -> pd.Series:
    """Light Savitzky-Golay denoising. Falls back to rolling median for short series."""
    x = s.astype(float)
    y = x.copy()

    valid = x.dropna()
    if len(valid) < 5:
        return x

    # Savitzky-Golay requires an odd window and window <= valid length.
    win = min(window, len(valid) if len(valid) % 2 == 1 else len(valid) - 1)
    if win <= polyorder + 2:
        return x.rolling(7, center=True, min_periods=1).median()
    if win % 2 == 0:
        win -= 1

    filled = interpolate_small_gaps(x).ffill().bfill()
    smoothed = savgol_filter(filled.values, window_length=win, polyorder=polyorder)
    y.loc[:] = smoothed
    y[x.isna()] = np.nan
    return y.rename(s.name)


def detrend_series(s: pd.Series, window: int = DETREND_WINDOW) -> pd.Series:
    """Remove slow trend using a centered rolling median."""
    x = s.astype(float)
    trend = x.rolling(window=window, center=True, min_periods=max(7, window // 4)).median()
    trend = trend.bfill().ffill()
    return (x - trend).rename(s.name)


def robust_zscore_series(s: pd.Series) -> pd.Series:
    """Robust per-query normalization using median and MAD."""
    x = s.astype(float)
    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)
        if pd.isna(std) or std == 0:
            return pd.Series(np.zeros(len(x)), index=x.index, name=x.name)
        return ((x - x.mean(skipna=True)) / std).rename(x.name)

    return (0.6745 * (x - med) / mad).rename(x.name)


def preprocess_panel(panel: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Apply the current preprocessing pipeline to all columns.

    Returns
    -------
    panel_preprocessed:
        daily panel after interpolation, denoising, detrending, and robust z-score.
    stages:
        intermediate panels useful for diagnostics and plotting.
    """
    filled = panel.apply(interpolate_small_gaps, axis=0)
    denoised = filled.apply(denoise_series, axis=0)
    detrended = denoised.apply(detrend_series, axis=0)
    normalized = detrended.apply(robust_zscore_series, axis=0)

    stages = {
        "filled": filled,
        "denoised": denoised,
        "detrended": detrended,
        "normalized": normalized,
    }
    return normalized, stages


## 3. Sliding SAX representation

The three retained tuning dimensions are implemented here:

- `window_size`: the local time horizon summarized by one SAX word;
- `step_size`: how often the window moves forward;
- `PAA word length w`: implemented as `n_segments`;
- `alphabet_size a`: number of vertical states.

For example, `window_size = 180`, `step = 30`, `n_segments = 24`, and `alphabet_size = 4` means each 180-day local episode is compressed into a 24-symbol word using four vertical levels.


In [5]:
def gaussian_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian SAX breakpoints for equiprobable symbolic bins."""
    return stats.norm.ppf(np.linspace(0, 1, alphabet_size + 1)[1:-1])


def valid_sax_setting(window_size: int, step: int, n_segments: int, alphabet_size: int) -> bool:
    """Keep settings interpretable and numerically valid."""
    if window_size < 2 or step < 1 or n_segments < 2 or alphabet_size < 2:
        return False
    if n_segments >= window_size:
        return False
    if window_size / n_segments < 3:
        return False
    return True


def panel_to_array_dict(panel: pd.DataFrame) -> dict[str, np.ndarray]:
    """Convert the normalized panel to compact NumPy arrays once."""
    out: dict[str, np.ndarray] = {}
    for col in panel.columns:
        x = panel[col].dropna().to_numpy(dtype=np.float32)
        if len(x) > 0:
            out[col] = x
    return out


def znormalize_windows_matrix(windows: np.ndarray) -> np.ndarray:
    """Vectorized z-normalization for all sliding windows of one series."""
    mu = windows.mean(axis=1, keepdims=True)
    sigma = windows.std(axis=1, keepdims=True)
    sigma[sigma == 0] = 1.0
    return (windows - mu) / sigma


def paa_matrix(z_windows: np.ndarray, n_segments: int) -> np.ndarray:
    """Vectorized PAA using cumulative sums for all windows at once."""
    n_windows, window_size = z_windows.shape
    bounds = np.linspace(0, window_size, n_segments + 1, dtype=np.int64)
    csum = np.concatenate(
        [np.zeros((n_windows, 1), dtype=z_windows.dtype), np.cumsum(z_windows, axis=1)],
        axis=1,
    )
    seg_sum = csum[:, bounds[1:]] - csum[:, bounds[:-1]]
    seg_len = (bounds[1:] - bounds[:-1]).astype(z_windows.dtype)
    return seg_sum / seg_len


def make_window_cache(
    array_dict: dict[str, np.ndarray],
    settings: list[dict[str, int]],
) -> dict[tuple[int, int], dict[str, np.ndarray]]:
    """Cache sliding windows for every distinct `(window_size, step)` pair."""
    cache: dict[tuple[int, int], dict[str, np.ndarray]] = {}
    pairs = sorted({(s["window_size"], s["step"]) for s in settings})

    for window_size, step in pairs:
        term_windows: dict[str, np.ndarray] = {}
        for term, x in array_dict.items():
            if len(x) < window_size:
                continue
            windows = sliding_window_view(x, window_size)[::step].astype(np.float32, copy=True)
            if len(windows) > 0:
                term_windows[term] = windows
        cache[(window_size, step)] = term_windows

    return cache


def build_sax_feature_matrix_fast(
    array_dict: dict[str, np.ndarray],
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int,
    spike_features: bool = INCLUDE_SPIKE_FEATURES,
    window_cache: dict[tuple[int, int], dict[str, np.ndarray]] | None = None,
) -> pd.DataFrame:
    """
    Fast query-level SAX profile construction.

    Feature design is unchanged:
    - mean symbolic level for each PAA segment across sliding windows;
    - distribution of symbols across the whole query;
    - optional spike features from the normalized preprocessed series.
    """
    if not valid_sax_setting(window_size, step, n_segments, alphabet_size):
        raise ValueError(
            f"Invalid SAX setting: window={window_size}, step={step}, "
            f"n_segments={n_segments}, alphabet={alphabet_size}"
        )

    breakpoints = gaussian_breakpoints(alphabet_size).astype(np.float32)
    cached_windows = None if window_cache is None else window_cache.get((window_size, step))
    feature_rows = {}

    iterator = cached_windows.items() if cached_windows is not None else array_dict.items()

    for term, obj in iterator:
        if cached_windows is not None:
            windows = obj
            x = array_dict[term]
        else:
            x = obj
            if len(x) < window_size:
                continue
            windows = sliding_window_view(x, window_size)[::step].astype(np.float32, copy=True)
            if len(windows) == 0:
                continue

        z = znormalize_windows_matrix(windows)
        seg_means = paa_matrix(z, n_segments)
        symbols = np.searchsorted(breakpoints, seg_means).astype(np.int16)

        row = {f"sax_seg_mean_{i + 1:02d}": float(v) for i, v in enumerate(symbols.mean(axis=0))}

        counts = np.bincount(symbols.ravel(), minlength=alphabet_size).astype(float)
        shares = counts / counts.sum()
        for a, share in enumerate(shares):
            row[f"symbol_share_{a}"] = float(share)

        if spike_features:
            row.update({
                "max_z": float(np.max(x)),
                "min_z": float(np.min(x)),
                "range_z": float(np.max(x) - np.min(x)),
                "p95_z": float(np.percentile(x, 95)),
                "p99_z": float(np.percentile(x, 99)),
                "spike_share_z2": float(np.mean(x > 2)),
                "spike_share_z3": float(np.mean(x > 3)),
            })

        feature_rows[term] = row

    return pd.DataFrame.from_dict(feature_rows, orient="index").dropna(axis=0, how="any")


# Backward-compatible alias.
def build_sax_feature_matrix(*args, **kwargs) -> pd.DataFrame:
    return build_sax_feature_matrix_fast(*args, **kwargs)


## 4. Tune the SAX representation

This stage tunes the SAX representation, not the final interpretation. It evaluates each setting using the best available KMeans score over a small `k` grid.

Recommended selection rule:

1. discard settings with too few usable terms;
2. prefer higher silhouette;
3. prefer lower Davies-Bouldin;
4. among close candidates, choose the simpler/more interpretable representation.

This avoids picking a very high-resolution SAX encoding that only wins because it compresses less.


In [6]:
def make_sax_grid(
    window_grid=WINDOW_GRID,
    step_fractions=STEP_FRACTIONS,
    paa_grid=PAA_WORD_LENGTH_GRID,
    alphabet_grid=ALPHABET_GRID,
) -> list[dict[str, int]]:
    """Construct valid SAX settings from the requested parameter families."""
    settings = []
    for window_size in window_grid:
        for frac in step_fractions:
            step = max(1, int(round(window_size * frac)))
            for n_segments in paa_grid:
                for alphabet_size in alphabet_grid:
                    if valid_sax_setting(window_size, step, n_segments, alphabet_size):
                        settings.append({
                            "window_size": int(window_size),
                            "step": int(step),
                            "n_segments": int(n_segments),
                            "alphabet_size": int(alphabet_size),
                        })
    return settings


def evaluate_clustering_for_features(
    features: pd.DataFrame,
    k_grid=K_GRID,
    random_state: int = RANDOM_STATE,
    n_init: int = TUNING_KMEANS_N_INIT,
    silhouette_sample_size: int | None = SILHOUETTE_SAMPLE_SIZE,
) -> pd.DataFrame:
    """
    Evaluate MiniBatchKMeans across k for one feature matrix.

    Silhouette is expensive because it uses pairwise distances. During tuning,
    this function samples rows when there are many terms.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    rows = []

    sample_size = None
    if silhouette_sample_size is not None and len(features) > silhouette_sample_size:
        sample_size = int(silhouette_sample_size)

    valid_ks = [k for k in k_grid if 2 <= k < len(features)]
    for k in valid_ks:
        labels = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=n_init,
            batch_size=min(512, len(features)),
            max_iter=100,
        ).fit_predict(X)

        rows.append({
            "k": int(k),
            "silhouette": float(silhouette_score(
                X,
                labels,
                sample_size=sample_size,
                random_state=random_state,
            )),
            "davies_bouldin": float(davies_bouldin_score(X, labels)),
            "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
        })

    return pd.DataFrame(rows)


def tune_sax_representation(
    array_dict: dict[str, np.ndarray],
    settings: list[dict[str, int]],
    k_grid=K_GRID,
    spike_features: bool = INCLUDE_SPIKE_FEATURES,
    min_terms: int = 10,
    use_window_cache: bool = USE_WINDOW_CACHE,
) -> pd.DataFrame:
    """Grid-search window, step, PAA word length, and alphabet size efficiently."""
    rows = []
    window_cache = make_window_cache(array_dict, settings) if use_window_cache else None

    for i, setting in enumerate(settings, start=1):
        try:
            features = build_sax_feature_matrix_fast(
                array_dict=array_dict,
                window_size=setting["window_size"],
                n_segments=setting["n_segments"],
                alphabet_size=setting["alphabet_size"],
                step=setting["step"],
                spike_features=spike_features,
                window_cache=window_cache,
            )
        except Exception as e:
            rows.append({**setting, "status": f"failed: {e}"})
            continue

        if len(features) < min_terms:
            rows.append({
                **setting,
                "status": "too_few_terms",
                "n_terms": len(features),
            })
            continue

        diag = evaluate_clustering_for_features(features, k_grid=k_grid)
        if diag.empty:
            rows.append({
                **setting,
                "status": "no_valid_k",
                "n_terms": len(features),
            })
            continue

        best = (
            diag.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True])
            .iloc[0]
        )

        rows.append({
            **setting,
            "status": "ok",
            "n_terms": len(features),
            "n_features": features.shape[1],
            "best_k": int(best["k"]),
            "silhouette": float(best["silhouette"]),
            "davies_bouldin": float(best["davies_bouldin"]),
            "calinski_harabasz": float(best["calinski_harabasz"]),
        })

        if i % 10 == 0:
            print(f"  evaluated {i}/{len(settings)} SAX settings")

    results = pd.DataFrame(rows)
    ok = results[results["status"] == "ok"].copy()
    not_ok = results[results["status"] != "ok"].copy()

    if not ok.empty:
        ok = ok.sort_values(
            ["silhouette", "davies_bouldin", "n_features"],
            ascending=[False, True, True],
        )

    return pd.concat([ok, not_ok], ignore_index=True)


## 5. Final clustering and candidate comparison

After selecting a SAX setting, this section builds the final feature matrix and compares a small number of cluster counts. For the clinic brief, prioritize a solution that is both statistically reasonable and easy to describe.


In [7]:
def run_kmeans_final(
    features: pd.DataFrame,
    k: int,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.Series, dict]:
    """Run final KMeans on scaled SAX features."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    km = KMeans(n_clusters=k, random_state=random_state, n_init=FINAL_KMEANS_N_INIT)
    labels_array = km.fit_predict(X)
    labels = pd.Series(labels_array, index=features.index, name="cluster")

    metrics = {
        "k": int(k),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
        "inertia": float(km.inertia_),
    }
    return labels, metrics


def run_ward_final(features: pd.DataFrame, k: int) -> tuple[pd.Series, np.ndarray, dict]:
    """Ward hierarchical clustering for comparison."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method="ward")
    labels_array = fcluster(Z, t=k, criterion="maxclust") - 1
    labels = pd.Series(labels_array, index=features.index, name="cluster")
    metrics = {
        "k": int(k),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
    }
    return labels, Z, metrics


def cluster_member_table(labels: pd.Series) -> pd.DataFrame:
    return (
        labels.rename_axis("term")
        .reset_index()
        .sort_values(["cluster", "term"])
        .reset_index(drop=True)
    )


def cluster_size_table(labels: pd.Series) -> pd.DataFrame:
    return (
        labels.value_counts()
        .sort_index()
        .rename_axis("cluster")
        .reset_index(name="n_terms")
    )


## 5b. Hierarchical clustering add-on

This optional block runs hierarchical clustering on the same tuned sliding-SAX feature matrix used by KMeans. It is useful as a robustness check because it does not rely on random centroid initialization and produces a dendrogram that can be shown in the clinic brief.

Recommended use:

- keep KMeans as the main scalable baseline;
- use Ward hierarchical clustering as the interpretable comparison;
- compare cluster sizes and validation metrics before deciding which result to present.

In [8]:
def run_hierarchical_grid(
    features: pd.DataFrame,
    k_grid=K_GRID,
    methods=("ward", "average", "complete"),
) -> pd.DataFrame:
    """
    Evaluate hierarchical clustering across linkage methods and k values.

    Notes:
    - Ward uses Euclidean geometry and usually works best with standardized continuous features.
    - Average/complete are included as robustness checks.
    - The returned table can be compared with KMeans tuning/final metrics.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    rows = []

    for method in methods:
        try:
            Z = linkage(X, method=method)
        except Exception as e:
            rows.append({"method": method, "status": f"failed: {e}"})
            continue

        for k in k_grid:
            if not (2 <= k < len(features)):
                continue

            labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1

            # Skip degenerate cuts that do not produce at least two clusters.
            if len(np.unique(labels_array)) < 2:
                continue

            rows.append({
                "method": method,
                "k": int(k),
                "n_clusters_observed": int(len(np.unique(labels_array))),
                "silhouette": float(silhouette_score(X, labels_array)),
                "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
                "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
                "status": "ok",
            })

    results = pd.DataFrame(rows)
    ok = results[results["status"] == "ok"].copy()
    not_ok = results[results["status"] != "ok"].copy()

    if not ok.empty:
        ok = ok.sort_values(
            ["silhouette", "davies_bouldin", "calinski_harabasz"],
            ascending=[False, True, False],
        )

    return pd.concat([ok, not_ok], ignore_index=True)


def run_hierarchical_final(
    features: pd.DataFrame,
    k: int,
    method: str = "ward",
) -> tuple[pd.Series, np.ndarray, dict]:
    """Run final hierarchical clustering with a chosen linkage method and k."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method=method)
    labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
    labels = pd.Series(labels_array, index=features.index, name="cluster")

    metrics = {
        "method": method,
        "k": int(k),
        "n_clusters_observed": int(len(np.unique(labels_array))),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
    }
    return labels, Z, metrics


In [9]:
def plot_cluster_average_curves(
    panel_norm: pd.DataFrame,
    labels: pd.Series,
    outpath: Path,
    title: str = "Cluster-average normalized attention curves",
):
    """Plot one average curve per cluster."""
    plt.figure(figsize=(12, 6))
    for c in sorted(labels.unique()):
        members = labels[labels == c].index.intersection(panel_norm.columns)
        if len(members) == 0:
            continue
        avg = panel_norm[members].mean(axis=1, skipna=True)
        plt.plot(avg.index, avg.values, linewidth=1.8, label=f"Cluster {c} (n={len(members)})")

    plt.axhline(0, linewidth=0.8)
    plt.title(title)
    plt.ylabel("Robust z-score after denoising and detrending")
    plt.xlabel("Date")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


def plot_dendrogram_for_features(features: pd.DataFrame, Z: np.ndarray, outpath: Path):
    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=features.index.tolist(), leaf_rotation=90, leaf_font_size=6)
    plt.title("Ward dendrogram on selected SAX features")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


## 6. Run the full current pipeline

Run this cell after checking the configuration paths above. It writes all key outputs to `OUTPUT_DIR`.


In [10]:
# -----------------------------------------------------------------------------
# Execute current pipeline
# -----------------------------------------------------------------------------

series_raw = load_all_series(DATA_DIR)
panel_raw = build_panel(series_raw)

quality = basic_quality_report(panel_raw)
quality.to_csv(OUTPUT_DIR / "01_post_stitch_quality_report.csv", index=False)

panel_norm, preprocessing_stages = preprocess_panel(panel_raw)
for name, stage in preprocessing_stages.items():
    stage.to_csv(OUTPUT_DIR / f"02_preprocess_{name}.csv")

array_norm = panel_to_array_dict(panel_norm)

settings = make_sax_grid()
print(f"Evaluating {len(settings)} SAX settings...")

tuning_results = tune_sax_representation(array_norm, settings)
tuning_results.to_csv(OUTPUT_DIR / "03_sax_window_step_paa_alphabet_tuning.csv", index=False)

ok_results = tuning_results[tuning_results["status"] == "ok"].copy()
if ok_results.empty:
    raise ValueError("No valid SAX tuning results. Check data length and parameter grid.")

selected = ok_results.iloc[0].to_dict()
selected_params = {
    "window_size": int(selected["window_size"]),
    "step": int(selected["step"]),
    "n_segments": int(selected["n_segments"]),
    "alphabet_size": int(selected["alphabet_size"]),
    "selected_k": int(selected["best_k"]),
    "spike_features": INCLUDE_SPIKE_FEATURES,
}

pd.DataFrame([selected_params]).to_csv(OUTPUT_DIR / "04_selected_sax_parameters.csv", index=False)

print("Selected SAX parameters:")
print(json.dumps(selected_params, indent=2))
print("\nTop 10 SAX settings:")
print(ok_results.head(10).to_string(index=False))

features = build_sax_feature_matrix_fast(
    array_dict=array_norm,
    window_size=selected_params["window_size"],
    n_segments=selected_params["n_segments"],
    alphabet_size=selected_params["alphabet_size"],
    step=selected_params["step"],
    spike_features=selected_params["spike_features"],
)
features.to_csv(OUTPUT_DIR / "05_selected_sax_feature_matrix.csv")

# Final KMeans using the selected k from representation tuning.
labels_kmeans, metrics_kmeans = run_kmeans_final(features, k=selected_params["selected_k"])
cluster_member_table(labels_kmeans).to_csv(OUTPUT_DIR / "06_kmeans_labels.csv", index=False)
cluster_size_table(labels_kmeans).to_csv(OUTPUT_DIR / "06_kmeans_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_kmeans]).to_csv(OUTPUT_DIR / "06_kmeans_metrics.csv", index=False)

# Hierarchical clustering comparison on the same selected SAX feature matrix.
hierarchical_results = run_hierarchical_grid(features, k_grid=K_GRID)
hierarchical_results.to_csv(OUTPUT_DIR / "07_hierarchical_tuning.csv", index=False)

best_hier = hierarchical_results[hierarchical_results["status"] == "ok"].iloc[0].to_dict()
labels_hier, Z_hier, metrics_hier = run_hierarchical_final(
    features,
    k=int(best_hier["k"]),
    method=str(best_hier["method"]),
)

cluster_member_table(labels_hier).to_csv(OUTPUT_DIR / "07_hierarchical_labels.csv", index=False)
cluster_size_table(labels_hier).to_csv(OUTPUT_DIR / "07_hierarchical_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_hier]).to_csv(OUTPUT_DIR / "07_hierarchical_metrics.csv", index=False)

# Also save Ward with the KMeans-selected k for direct one-to-one comparison.
labels_ward, Z_ward, metrics_ward = run_ward_final(features, k=selected_params["selected_k"])
cluster_member_table(labels_ward).to_csv(OUTPUT_DIR / "07_ward_same_k_labels.csv", index=False)
cluster_size_table(labels_ward).to_csv(OUTPUT_DIR / "07_ward_same_k_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_ward]).to_csv(OUTPUT_DIR / "07_ward_same_k_metrics.csv", index=False)

plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_kmeans,
    outpath=OUTPUT_DIR / "08_kmeans_cluster_average_curves.png",
    title="KMeans clusters on tuned sliding-SAX features",
)
plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_hier,
    outpath=OUTPUT_DIR / "08_hierarchical_cluster_average_curves.png",
    title="Hierarchical clusters on tuned sliding-SAX features",
)
plot_dendrogram_for_features(features, Z_hier, OUTPUT_DIR / "08_hierarchical_dendrogram.png")
plot_dendrogram_for_features(features, Z_ward, OUTPUT_DIR / "08_ward_same_k_dendrogram.png")

print("\nKMeans metrics:")
print(metrics_kmeans)
print("\nBest hierarchical metrics:")
print(metrics_hier)
print("\nWard same-k metrics:")
print(metrics_ward)
print("\nTop hierarchical settings:")
print(hierarchical_results.head(10).to_string(index=False))
print(f"\nOutputs written to: {OUTPUT_DIR}")


Loaded 492 stitched series.
Evaluating 76 SAX settings...
  evaluated 10/76 SAX settings
  evaluated 20/76 SAX settings
  evaluated 30/76 SAX settings
  evaluated 40/76 SAX settings
  evaluated 50/76 SAX settings
  evaluated 60/76 SAX settings
  evaluated 70/76 SAX settings
Selected SAX parameters:
{
  "window_size": 60,
  "step": 15,
  "n_segments": 16,
  "alphabet_size": 6,
  "selected_k": 2,
  "spike_features": true
}

Top 10 SAX settings:
 window_size  step  n_segments  alphabet_size status  n_terms  n_features  best_k  silhouette  davies_bouldin  calinski_harabasz
          60    15          16              6     ok      492          29       2    0.722540        0.466717         172.771421
          60    15          16              4     ok      492          27       2    0.713733        0.481614         163.004291
          60    15          16              5     ok      492          28       3    0.711615        0.475617         244.753768
          60    15          12       

## 7. Interpretation notes for reporting

Use the following outputs for the clinic brief:

- `03_sax_window_step_paa_alphabet_tuning.csv`  
  Shows how the retained SAX parameters performed.

- `04_selected_sax_parameters.csv`  
  Documents the chosen `window_size`, `step`, `PAA word length w`, and `alphabet_size a`.

- `05_selected_sax_feature_matrix.csv`  
  Query-level feature matrix used for clustering.

- `06_kmeans_labels.csv` and `06_kmeans_cluster_sizes.csv`  
  Main cluster assignments and cluster sizes.

- `07_hierarchical_tuning.csv`  
  Hierarchical clustering comparison across Ward, average, and complete linkage.

- `07_hierarchical_labels.csv`, `07_hierarchical_cluster_sizes.csv`, and `07_hierarchical_metrics.csv`  
  Final hierarchical clustering outputs.

- `08_kmeans_cluster_average_curves.png` and `08_hierarchical_cluster_average_curves.png`  
  Best figures for comparing cluster-average attention shapes.

### Revision summary

Compared with the previous notebook:

1. The pipeline now starts from the current `data_events/<query>/stitched/` structure.
2. The preprocessing stage is explicit and saved to disk.
3. SAX tuning now searches over:
   - `window_size`,
   - `step`,
   - `n_segments` / PAA word length `w`,
   - `alphabet_size` / alphabet resolution `a`.
4. Alphabet tuning is no longer the only SAX tuning step.
5. Consensus-threshold and Leiden/Louvain resolution tuning are excluded because they are not part of the current KMeans/hierarchical workflow.
6. The external GitHub repositories are referenced as methodological guidance, not as code dependencies.


### Efficiency revision summary

The computationally expensive part is the grid search over sliding-SAX representations. The revised code now caches NumPy arrays and sliding windows, vectorizes PAA, and uses sampled silhouette scoring during tuning. This should make the tuning cell much faster without changing the conceptual pipeline or the final output files.
